In [ ]:
# Cell 1: imports, paths, basic sanity

import numpy as np
import cv2
from pathlib import Path

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# double checking env
from skimage.feature import hog, local_binary_pattern
try:
    from skimage.feature import graycomatrix, graycoprops
except ImportError:
    from skimage.feature.texture import graycomatrix, graycoprops

from skimage.measure import label, regionprops

from joblib import Parallel, delayed
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# double checking my env
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not found – XGBClassifier section will be skipped.")

CLASSES = [
    "background",
    "missing_hole",
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper",
]

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
PATCH_ROOT = REPO_ROOT / "data" / "patches_sw"
FEATURE_ROOT = REPO_ROOT / "data" / "features"
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

print("Repo root :", REPO_ROOT)
print("PATCH_ROOT:", PATCH_ROOT)
print("Exists?   :", PATCH_ROOT.exists())

for split in ["train", "valid", "test"]:
    print(f"\nSPLIT: {split}")
    split_dir = PATCH_ROOT / split
    for cls in CLASSES:
        cls_dir = split_dir / cls
        cnt = len(list(cls_dir.glob("*.jpg")))
        print(f"{cls:15s} -> {cnt}")

In [ ]:
# Cell 2: show a random patch for sanity

def show_random_patch(split="train", seed=42):
    rng = np.random.default_rng(seed)
    split_dir = PATCH_ROOT / split
    cls = rng.choice(CLASSES)
    cls_dir = split_dir / cls
    paths = sorted(cls_dir.glob("*.jpg"))
    if not paths:
        print("No patches for", split, cls)
        return
    p = rng.choice(paths)
    img_bgr = cv2.imread(str(p))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    plt.imshow(img_rgb)
    plt.axis("off")
    plt.title(f"{split} / {cls}\n{p.name}")
    plt.show()

show_random_patch("train")


In [ ]:
# Cell 3: feature helper functions

def split_into_grid(img, n_rows=2, n_cols=2):
    h, w = img.shape
    r_step, c_step = h // n_rows, w // n_cols
    patches = []
    for r in range(n_rows):
        for c in range(n_cols):
            patch = img[r*r_step:(r+1)*r_step, c*c_step:(c+1)*c_step]
            patches.append(patch)
    return patches

# feature time!! ---------------------------------------------------

def zoned_hog(img_gray, n_rows=2, n_cols=2,
              orientations=8, pixels_per_cell=(8, 8),
              cells_per_block=(1, 1)):
    patches = split_into_grid(img_gray, n_rows=n_rows, n_cols=n_cols)
    feats = []
    for p in patches:
        h = hog(
            p,
            orientations=orientations,
            pixels_per_cell=pixels_per_cell,
            cells_per_block=cells_per_block,
            block_norm="L2-Hys",
            feature_vector=True,
        )
        feats.append(h.astype(np.float32))
    return np.concatenate(feats)


LBP_P = 8
LBP_R = 1


def lbp_hist(img_gray):
    lbp = local_binary_pattern(img_gray, P=LBP_P, R=LBP_R, method="uniform")
    hist, _ = np.histogram(
        lbp,
        bins=np.arange(0, LBP_P + 3),
        range=(0, LBP_P + 2),
        density=True,
    )
    return hist.astype(np.float32)


def glcm_features(img_gray, levels=32):
    img_f = img_gray.astype(np.float32) / 255.0
    img_f = np.clip(img_f, 0.0, 1.0)
    img_q = (img_f * (levels - 1)).round().astype(np.uint8)

    distances = [1]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(
        img_q,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True,
    )
    props = ["contrast", "homogeneity", "ASM"]
    feats = []
    for prop in props:
        vals = graycoprops(glcm, prop)
        feats.append(vals.ravel())
    return np.concatenate(feats).astype(np.float32)  


def zoned_lbp_glcm(img_gray, n_rows=2, n_cols=2):
    patches = split_into_grid(img_gray, n_rows=n_rows, n_cols=n_cols)
    lbp_all, glcm_all = [], []
    for p in patches:
        lbp_all.append(lbp_hist(p))
        glcm_all.append(glcm_features(p))
    return np.concatenate(lbp_all), np.concatenate(glcm_all)


def edge_stats(img_gray):
    edges = cv2.Canny(img_gray, 50, 150)
    edge_ratio = edges.mean().astype(np.float32)

    sobelx = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=3)
    angles = np.arctan2(sobely, sobelx)
    if np.any(edges):
        orient_std = np.std(angles[edges > 0]).astype(np.float32)
    else:
        orient_std = np.float32(0.0)
    return np.array([edge_ratio, orient_std], dtype=np.float32)


def color_stats(img_rgb):
    mean = img_rgb.mean(axis=(0, 1)) / 255.0
    std = img_rgb.std(axis=(0, 1)) / 255.0
    return np.concatenate([mean, std]).astype(np.float32)


def pad_shape_features(img_gray):
    _, th = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if th.mean() > 127:
        th = 255 - th

    lbl = label(th > 0)
    props = regionprops(lbl)
    if not props:
        return np.zeros(4, dtype=np.float32)

    r = max(props, key=lambda p: p.area)
    area = float(r.area)
    perimeter = float(r.perimeter) if r.perimeter is not None else 0.0
    ecc = float(r.eccentricity)
    solidity = float(r.solidity)
    return np.array([area, perimeter, ecc, solidity], dtype=np.float32)

# paper based features --------------------------------------------
def fast_features(img_gray):
    fast = cv2.FastFeatureDetector_create(threshold=20, nonmaxSuppression=True)
    kpts = fast.detect(img_gray, None)
    n = len(kpts)
    if n > 0:
        responses = np.array([kp.response for kp in kpts], dtype=np.float32)
        resp_mean = float(responses.mean())
    else:
        resp_mean = 0.0
    return np.array([n, resp_mean], dtype=np.float32)


def orb_features(img_gray):
    orb = cv2.ORB_create(nfeatures=256, fastThreshold=20)
    kpts = orb.detect(img_gray, None)
    n = len(kpts)
    if n > 0:
        sizes = np.array([kp.size for kp in kpts], dtype=np.float32)
        size_mean = float(sizes.mean())
    else:
        size_mean = 0.0
    return np.array([n, size_mean], dtype=np.float32)


def fft_features(img_gray):
    f = np.fft.fft2(img_gray.astype(np.float32))
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)

    h, w = mag.shape
    cy, cx = h // 2, w // 2
    r = min(h, w) // 8
    center = mag[cy-r:cy+r, cx-r:cx+r].sum()
    total = mag.sum() + 1e-6

    low_ratio = float(center / total)
    high_ratio = float((total - center) / total)
    return np.array([low_ratio, high_ratio], dtype=np.float32)


def contrast_features(img_gray):
    std = float(img_gray.std() / 255.0)
    rng = float((img_gray.max() - img_gray.min()) / 255.0)
    return np.array([std, rng], dtype=np.float32)


In [ ]:
# Cell 4: extract_features_multi + probe lengths!

def extract_features_multi(img_rgb):

    # normalize to 64x64 for consistent feature dims
    img_rgb = cv2.resize(img_rgb, (64, 64), interpolation=cv2.INTER_AREA)
    img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # HOG pyramid
    hog_feats = []
    for size in [64, 32, 16]:
        img_resized = cv2.resize(img_gray, (size, size), interpolation=cv2.INTER_AREA)
        hog_feats.append(zoned_hog(img_resized))
    hog_feats = np.concatenate(hog_feats).astype(np.float32)

    # Non-HOG features
    lbp_z, glcm_z = zoned_lbp_glcm(img_gray)
    edge_feat = edge_stats(img_gray)
    color_feat = color_stats(img_rgb)
    pad_shape = pad_shape_features(img_gray)
    fast_feat = fast_features(img_gray)
    orb_feat = orb_features(img_gray)
    fft_feat = fft_features(img_gray)
    contrast_feat = contrast_features(img_gray)

    other_feats = np.concatenate([
        lbp_z, glcm_z, edge_feat, color_feat,
        pad_shape, fast_feat, orb_feat, fft_feat, contrast_feat
    ]).astype(np.float32)

    return hog_feats, other_feats


# probe one patch to get dims + block breakdown
probe_split_dir = PATCH_ROOT / "train"
probe_cls = CLASSES[0]
probe_path = next((probe_split_dir / probe_cls).glob("*.jpg"))

img_bgr = cv2.imread(str(probe_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

hog_probe, other_probe = extract_features_multi(img_rgb)
HOG_RAW_LEN = len(hog_probe)
OTHER_LEN = len(other_probe)

img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
lbp_z, glcm_z = zoned_lbp_glcm(img_gray)
EDGE_LEN = len(edge_stats(img_gray))
COLOR_LEN = len(color_stats(img_rgb))
PAD_LEN = len(pad_shape_features(img_gray))
FAST_LEN = len(fast_features(img_gray))
ORB_LEN = len(orb_features(img_gray))
FFT_LEN = len(fft_features(img_gray))
CONTRAST_LEN = len(contrast_features(img_gray))
LBP_LEN = len(lbp_z)
GLCM_LEN = len(glcm_z)

print("HOG_RAW_LEN :", HOG_RAW_LEN)
print("OTHER_LEN   :", OTHER_LEN)
print("TOTAL (raw) :", HOG_RAW_LEN + OTHER_LEN)

print("\nOther block lengths:")
print("LBP     :", LBP_LEN)
print("GLCM    :", GLCM_LEN)
print("EDGE    :", EDGE_LEN)
print("COLOR   :", COLOR_LEN)
print("PAD     :", PAD_LEN)
print("FAST    :", FAST_LEN)
print("ORB     :", ORB_LEN)
print("FFT     :", FFT_LEN)
print("CONTRAST:", CONTRAST_LEN)

assert OTHER_LEN == (LBP_LEN + GLCM_LEN + EDGE_LEN + COLOR_LEN +
                     PAD_LEN + FAST_LEN + ORB_LEN + FFT_LEN + CONTRAST_LEN)


In [ ]:
# Cell 5: build feature matrices and save to disk

def load_patch(path):
    img_bgr = cv2.imread(str(path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    hog_f, other_f = extract_features_multi(img_rgb)
    feat = np.concatenate([hog_f, other_f])
    label_name = path.parent.name
    label_idx = CLASSES.index(label_name)
    return feat, label_idx

# helping for memory to send across users
def build_feature_matrix(split, n_jobs=4):
    split_dir = PATCH_ROOT / split
    all_feat = []
    all_label = []

    for cls in CLASSES:
        cls_dir = split_dir / cls
        paths = sorted(cls_dir.glob("*.jpg"))
        print(f"{split:5s} | {cls:15s}: {len(paths)} patches")

        if not paths:
            continue

        results = Parallel(n_jobs=n_jobs)(
            delayed(load_patch)(p) for p in tqdm(paths, leave=False)
        )
        for f, y in results:
            all_feat.append(f)
            all_label.append(y)

    X = np.vstack(all_feat).astype(np.float32)
    y = np.array(all_label, dtype=np.int32)
    print(f"{split} feature matrix: {X.shape}")
    return X, y


# Toggle this depending on whether you already saved features
RECOMPUTE_FEATURES = True

if RECOMPUTE_FEATURES:
    X_train_raw, y_train = build_feature_matrix("train", n_jobs=4)
    X_val_raw,   y_val   = build_feature_matrix("valid", n_jobs=4)
    X_test_raw,  y_test  = build_feature_matrix("test",  n_jobs=4)

    np.savez_compressed(
        FEATURE_ROOT / "pcb_features_raw.npz",
        X_train=X_train_raw, y_train=y_train,
        X_val=X_val_raw,     y_val=y_val,
        X_test=X_test_raw,   y_test=y_test,
    )
else:
    data = np.load(FEATURE_ROOT / "pcb_features_raw.npz")
    X_train_raw = data["X_train"]
    X_val_raw   = data["X_val"]
    X_test_raw  = data["X_test"]
    y_train     = data["y_train"]
    y_val       = data["y_val"]
    y_test      = data["y_test"]

print("Train raw:", X_train_raw.shape)
print("Val raw  :", X_val_raw.shape)
print("Test raw :", X_test_raw.shape)


In [ ]:
# Cell 6: HOG PCA + final feature vectors

def split_hog_other(X):
    hog = X[:, :HOG_RAW_LEN]
    other = X[:, HOG_RAW_LEN:]
    return hog, other

Xtr_hog, Xtr_other = split_hog_other(X_train_raw)
Xv_hog,  Xv_other  = split_hog_other(X_val_raw)
Xte_hog, Xte_other = split_hog_other(X_test_raw)

print("Train HOG/raw :", Xtr_hog.shape, Xtr_other.shape)

HOG_PCA_DIM = 64

pca_hog = PCA(n_components=HOG_PCA_DIM, whiten=True, random_state=42)
Xtr_hog_pca = pca_hog.fit_transform(Xtr_hog)
Xv_hog_pca  = pca_hog.transform(Xv_hog)
Xte_hog_pca = pca_hog.transform(Xte_hog)

X_train = np.hstack([Xtr_hog_pca, Xtr_other])
X_val   = np.hstack([Xv_hog_pca,  Xv_other])
X_test  = np.hstack([Xte_hog_pca, Xte_other])

print("Final features:")
print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

np.savez_compressed(
    FEATURE_ROOT / "pcb_features_pca.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)


In [ ]:
# Cell 7: basic EDA --> (label distribution + PCA 2D viz)

# label distribution --> train set
train_counts = pd.Series(y_train).value_counts().sort_index()
train_counts.index = [CLASSES[i] for i in train_counts.index]

plt.figure(figsize=(8,4))
train_counts.plot(kind="bar")
plt.ylabel("Count")
plt.title("Train label distribution (patches)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# PCA to 2D for visualization on a sampled subset
N_VIZ = 8000
idx = np.random.choice(len(X_train), size=min(N_VIZ, len(X_train)), replace=False)
X_viz = X_train[idx]
y_viz = y_train[idx]

scaler_viz = StandardScaler()
X_viz_scaled = scaler_viz.fit_transform(X_viz)

pca_viz = PCA(n_components=2, random_state=42)
X_viz_2d = pca_viz.fit_transform(X_viz_scaled)

plt.figure(figsize=(7,6))
for i, cls_name in enumerate(CLASSES):
    mask = (y_viz == i)
    plt.scatter(
        X_viz_2d[mask, 0],
        X_viz_2d[mask, 1],
        s=5,
        alpha=0.5,
        label=cls_name,
    )
plt.legend(markerscale=3, fontsize=8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA (2D) of engineered features --> subset")
plt.tight_layout()
plt.show()


In [ ]:
# EDA-1 --> label distribution for train/valid/test

def label_counts(y):
    s = pd.Series(y).value_counts().sort_index()
    s.index = [CLASSES[i] for i in s.index]
    return s

train_counts = label_counts(y_train)
val_counts   = label_counts(y_val)
test_counts  = label_counts(y_test)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

train_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Train")
axes[0].set_ylabel("Patch count")
axes[0].tick_params(axis="x", rotation=45)

val_counts.plot(kind="bar", ax=axes[1])
axes[1].set_title("Valid")
axes[1].tick_params(axis="x", rotation=45)

test_counts.plot(kind="bar", ax=axes[2])
axes[2].set_title("Test")
axes[2].tick_params(axis="x", rotation=45)

fig.suptitle("Label distribution across splits", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# EDA-2: PCA scree plot on engineered features --> train set

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# number of components to show
N_SCRE = 50  

scaler_eda = StandardScaler()
X_train_scaled_eda = scaler_eda.fit_transform(X_train)

pca_full = PCA(n_components=N_SCRE, random_state=42)
pca_full.fit(X_train_scaled_eda)

explained = pca_full.explained_variance_ratio_
cum_explained = np.cumsum(explained)

plt.figure(figsize=(7,4))
plt.plot(range(1, N_SCRE+1), cum_explained, marker="o")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA scree plot (engineered features, train)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()